# MQT QMAP pipeline for a zoned neutral-atom architecture

The source circuit and its native-gate transpilation live in reusable Python modules. This notebook selects the circuit and architecture, runs the compiler, and exports visualization files.

## Build and transpile the circuit

In [ ]:
from circuit import (
    build_rotated_surface_code_circuit, 
    build_blockwise_cz_verification_circuit, 
    build_17_qubit_surface_code_circuit,
    build_steane_code_circuit,
    build_steane_blockwise_cz_verification_circuit
)
from transpiler import transpile_to_native

source_circuit = build_blockwise_cz_verification_circuit(
    # distance=3,
    # max_ancillas=2,
    # max_timeout=60,
)
native_circuit = transpile_to_native(source_circuit)

cz_depth = native_circuit.depth(
    filter_function=lambda instruction: instruction.operation.name == "cz"
)

print("Source operations:", source_circuit.count_ops())
print("Native operations:", native_circuit.count_ops())
print("CZ depth / global Rydberg pulses:", cz_depth)

In [ ]:
source_circuit.draw(output="mpl", fold=-1)

### Native gate set

- Global $R_Y$
- Local $R_Z$
- Local $CZ$

In [ ]:
native_circuit.draw(output="mpl", fold=-1)

## Define the neutral-atom architecture

In [ ]:
from mqt.qmap.na.zoned import ZonedNeutralAtomArchitecture

arch = ZonedNeutralAtomArchitecture.from_json_string("""{
  "name": "Architecture with one entanglement and one storage zone",
  "operation_duration": {"rydberg_gate": 0.36, "single_qubit_gate": 52, "atom_transfer": 15},
  "operation_fidelity": {"rydberg_gate": 0.995, "single_qubit_gate": 0.9997, "atom_transfer": 0.999},
  "qubit_spec": {"T": 1.5e6},
  "storage_zones": [{
    "zone_id": 0,
    "slms": [{"id": 0, "site_separation": [12, 10], "r": 2, "c": 10, "location": [0, 0]}],
    "offset": [0, 0],
    "dimension": [108, 10]
  }],
  "entanglement_zones": [{
    "zone_id": 0,
    "slms": [
      {"id": 1, "site_separation": [12, 10], "r": 9, "c": 10, "location": [0, 20]},
      {"id": 2, "site_separation": [12, 10], "r": 9, "c": 10, "location": [2, 20]}
    ],
    "offset": [2, 20],
    "dimension": [110, 80]
  }],
  "aods": [{"id": 0, "site_separation": 2, "r": 10, "c": 10}],
  "rydberg_range": [[[0, 18], [114, 100]]]
}""")

## Compile

The routing-aware compiler is used by default. The routing-agnostic alternative remains available in the same cell.

In [ ]:
from mqt.qmap.na.zoned import RoutingAgnosticCompiler, RoutingAwareCompiler

compiler = RoutingAwareCompiler(arch)
# compiler = RoutingAgnosticCompiler(arch)

In [ ]:
from mqt.core import load

mqt_circuit = load(native_circuit)
compiled_program = compiler.compile(mqt_circuit)
print(compiled_program)

## Export NAViz inputs

In [ ]:
from pathlib import Path

output_dir = Path("visualisation_instructions")
output_dir.mkdir(parents=True, exist_ok=True)

arch.to_namachine_file(str(output_dir / "zoned_RSC_Steane_graph.namachine"))
(output_dir / "zoned_RSC_Steane_graph.naviz").write_text(str(compiled_program), encoding="utf-8")

## Next steps

Additional source circuits can be added as new builder functions in `circuit.py`. The architecture remains explicit in this notebook so that zoned configurations can be changed independently of circuit construction and transpilation.

`ZonedNeutralAtomArchitecture` requires at least one storage zone, so a purely monolithic configuration cannot be represented by simply omitting it.